# KYB triage: a walkthrough

A written onboarding rulebook, applied the same way to every application.
This notebook runs the real engine on 13 synthetic applications and shows:

1. what the rulebook says and who signed it off
2. one application traced rule by rule
3. the whole queue triaged, with the three KPIs
4. an MLRO evidence pack
5. the impact of a rulebook change *before* sign-off
6. overrides, and why some are refused
7. the audit log, and what happens when someone edits it

No model, no network. Same inputs, same rulebook, same date: same answer.

In [1]:
import copy, json, sys
from pathlib import Path
from IPython.display import Markdown, display

sys.path.insert(0, ".")
from kyb import rulebook, audit, cli, kpi
from kyb.engine import assess
from kyb.impact import impact

AS_OF = "2026-09-23T10:00:00Z"
rb = rulebook.load()
cases = [json.loads(p.read_text()) for p in sorted(Path("data/cases").glob("*.json"))]
m = rb["meta"]
print(f"{m['name']} v{m['version']} ({m['status']})")
print(f"Approved by {m['approved_by']} on {m['approved_on']}; next review by {m['next_review_by']}")
print(f"{len(rb['rules'])} rules, each citing a clause in {m['policy']}")
print(f"Rulebook fingerprint: {rb['_hash'][:16]}…")

Sample KYB Onboarding Rulebook v1.0.0 (approved)
Approved by Group MLRO (sample sign-off) on 2026-08-28; next review by 2027-02-28
18 rules, each citing a clause in policy/aml_policy.md
Rulebook fingerprint: e1a4006f40aa0db3…


## 1. One application, rule by rule

Northgate Analytics has two screening alerts: its director shares a name with a sanctioned person,
and the company name loosely matches an adverse-media record. Neither needs a human.

In [2]:
d = assess(next(c for c in cases if c["case_id"] == "C006"), rb, AS_OF)
print(d["outcome"], "|", d["reasons"][0], "\n")
for h in d["hits"]:
    print(f"{h['list']:<14} {h['subject']:<24} score {h['match_score']:.2f}  →  {h['state']}: {h['reason']}")
print()
for r in d["rules"]:
    print(f"{r['id']:<9} {'FIRED' if r['fired'] else 'pass ':<6} {r['name']}")

AUTO_APPROVE | No rule fired; low risk and eligible for auto-approval (POL-7.1) 

sanctions      Daniel Okafor            score 1.00  →  FALSE_POSITIVE: identifiers disagree: dob, country
adverse_media  Northgate Analytics Ltd  score 0.68  →  FALSE_POSITIVE: match score 0.68 below 0.75 and no identifier agrees

R-DOC-01  pass   Required documents present
R-DOC-02  pass   Documents recent enough to rely on
R-JUR-01  pass   No prohibited jurisdiction
R-JUR-02  pass   High-risk jurisdiction needs EDD
R-JUR-03  pass   Every jurisdiction is rated
R-SEC-01  pass   No prohibited sector
R-SEC-02  pass   Sector is rated
R-VOL-01  pass   Expected volume under escalation threshold
R-UBO-01  pass   UBOs identified and verified
R-UBO-02  pass   Ownership reconciles to 100%
R-UBO-03  pass   Ownership chain within depth limit
R-UBO-04  pass   No bearer shares or nominees
R-SCR-01  pass   No confirmed sanctions match
R-SCR-02  pass   Confirmed PEP needs EDD
R-SCR-03  pass   High-severity adverse media

## 2. The whole queue

In [3]:
decisions = cli.triage("data/cases", AS_OF, "output")
for d in decisions:
    print(f"{d['case_id']}  {d['legal_name']:<30} {d['outcome']:<16} {d['risk']['band']:<7} {' '.join(d['deciding_rules'])}")

C001  Brightline Software Ltd        AUTO_APPROVE     low     
C002  Harbour Freight Traders Ltd    ANALYST_REVIEW   medium  R-RISK-01
C003  Kestrel Design Studio Ltd      INCOMPLETE       low     R-DOC-01 R-DOC-02
C004  Orbit Digital Exchange Ltd     MLRO_ESCALATION  high    R-JUR-02 R-RISK-01
C005  Meridian Holdings Ltd          REJECT           low     R-JUR-01
C006  Northgate Analytics Ltd        AUTO_APPROVE     low     
C007  Coastal Metals Trading Ltd     MLRO_ESCALATION  medium  R-SCR-01
C008  Aster Payments Ltd             MLRO_ESCALATION  high    R-SCR-02 R-RISK-01
C009  Lumen Retail Co Ltd            INCOMPLETE       low     R-UBO-01 R-UBO-02
C010  Fairwind Ventures Ltd          MLRO_ESCALATION  low     R-UBO-04
C011  Silverline Gaming Ltd          REJECT           low     R-SEC-01
C012  Pinecrest Components Ltd       ANALYST_REVIEW   low     R-SCR-05
C013  Tidewater Labs Ltd             AUTO_APPROVE     low     


In [4]:
display(Markdown(Path("output/kpi.md").read_text()))

# KPI report

Rulebook v1.0.0 · 13 applications

| Outcome | Cases |
|---|---|
| AUTO_APPROVE | 3 |
| ANALYST_REVIEW | 2 |
| INCOMPLETE | 2 |
| MLRO_ESCALATION | 4 |
| REJECT | 2 |

## 1. KYB submission to trading-ready under 24 hours (low and medium risk)

9 applications scored low or medium risk.

- **3** approved automatically at submission, with no one touching them.
- **2** routed to analyst review with an 8-hour SLA, inside the 24-hour target if the SLA is met.
- **2** waiting on client information; outreach drafted, clock depends on the client.
- **2** escalated to the MLRO by a specific trigger despite a low or medium score.

Auto-approval is currently limited to bands: low. Extending it to medium risk is a
one-line rulebook change, and needs MLRO sign-off (POL-7.1).

## 2. Alerts cleared by rule rather than by hand

6 screening alerts. **5 (83%) resolved by rule**:
3 cleared as false positives, 2 confirmed.
1 left for an analyst; the pack shows which identifiers were missing or disagreed.

## 3. MLRO can decide without asking for more information

4 cases escalated. **3 of 4 (75%)
decision-ready**: documents complete and every screening hit resolved or explained.


## 3. An MLRO evidence pack

Aster Payments has a director who is a confirmed PEP. The pack gives the MLRO the decision,
the reason, every rule checked, how each alert was resolved, and a sign-off block.

In [5]:
display(Markdown(Path("output/packs/C008.md").read_text()))

# Evidence pack: C008 Aster Payments Ltd

| Field | Value |
|---|---|
| Outcome | **MLRO_ESCALATION**: MLRO decision required |
| Next action owner | Group MLRO |
| Next action due | 2026-09-24T10:00:00+00:00 (SLA 24h) |
| Risk | 70 (high) |
| Decision-ready for MLRO | yes |
| Submitted | 2026-09-23T08:30:00Z |
| Assessed as of | 2026-09-23T10:00:00+00:00 |
| Rulebook | v1.0.0, sha256 e1a4006f40aa0db3… |
| Application hash | sha256 a9682e45d9536573… |

## Why

- R-SCR-02 Confirmed PEP needs EDD: pep hit on Helena Varga (director) vs 'Helena Varga': match score 0.96 and identifiers agree: dob, country
- R-RISK-01 Risk band eligible for auto-approval: score 70 is high risk: not eligible for auto-approval

## Open items

None.

## Risk score

| Factor | Value | Points | Policy |
|---|---|---|---|
| Jurisdiction (highest rated) | XB (low) | 0 | POL-3.4 |
| Sector | money_services (high) | 30 | POL-4.2 |
| Company age | 80 months | 0 | POL-4.4 |
| Expected monthly volume | USD 800,000 | 10 | POL-4.3 |
| Confirmed PEP | yes | 30 | POL-6.4 |
| **Total** | **high** | **70** |  |

## Every rule checked

| Rule | Check | Result | Effect | Policy | Detail |
|---|---|---|---|---|---|
| R-DOC-01 | Required documents present | pass |  | POL-2.1 | all 5 required documents held |
| R-DOC-02 | Documents recent enough to rely on | pass |  | POL-2.2 | all dated documents within limits |
| R-JUR-01 | No prohibited jurisdiction | pass |  | POL-3.1 | none prohibited |
| R-JUR-02 | High-risk jurisdiction needs EDD | pass |  | POL-3.2 | none high-risk |
| R-JUR-03 | Every jurisdiction is rated | pass |  | POL-3.3 | all rated |
| R-SEC-01 | No prohibited sector | pass |  | POL-4.1 | sector 'money_services' permitted |
| R-SEC-02 | Sector is rated | pass |  | POL-4.5 | sector 'money_services' rated |
| R-VOL-01 | Expected volume under escalation threshold | pass |  | POL-4.3 | expected USD 800,000/month |
| R-UBO-01 | UBOs identified and verified | pass |  | POL-5.1 | 1 UBO(s) identified and verified |
| R-UBO-02 | Ownership reconciles to 100% | pass |  | POL-5.2 | declared ownership totals 100% |
| R-UBO-03 | Ownership chain within depth limit | pass |  | POL-5.3 | 1 layer(s) |
| R-UBO-04 | No bearer shares or nominees | pass |  | POL-5.4 | none declared |
| R-SCR-01 | No confirmed sanctions match | pass |  | POL-6.3 | no confirmed sanctions match |
| R-SCR-02 | Confirmed PEP needs EDD | FIRED | escalate | POL-6.4 | pep hit on Helena Varga (director) vs 'Helena Varga': match score 0.96 and identifiers agree: dob, country |
| R-SCR-03 | High-severity adverse media | pass |  | POL-6.5 | none |
| R-SCR-04 | Other confirmed adverse media | pass |  | POL-6.5 | none |
| R-SCR-05 | No unresolved potential matches | pass |  | POL-6.6 | all hits resolved by rule |
| R-RISK-01 | Risk band eligible for auto-approval | FIRED | escalate | POL-7.1 | score 70 is high risk: not eligible for auto-approval |

## Screening

| List | Subject | Role | Matched name | Score | Result | How it was resolved |
|---|---|---|---|---|---|---|
| pep | Helena Varga | director | Helena Varga | 0.96 | CONFIRMED | match score 0.96 and identifiers agree: dob, country |

## Entity

| Field | Value |
|---|---|
| Legal name | Aster Payments Ltd |
| Registration number | REG-008-2020 |
| Entity type | company |
| Incorporated | 2020-01-15 in XB |
| Operates in | XB |
| Sector | money_services |
| Expected monthly volume | USD 800,000 |

## Ownership

| Owner | Type | Ownership | ID verified | Nationality |
|---|---|---|---|---|
| Samuel Ortiz | individual | 100% | yes | XB |

Ownership layers: 1. Bearer shares: no. Nominee shareholders: no.

## Documents held

| Document | Dated |
|---|---|
| certificate_of_incorporation | 2020-01-15 |
| register_of_directors | 2026-03-15 |
| register_of_shareholders | 2026-03-15 |
| proof_of_address | 2026-08-10 |
| ubo_declaration | 2026-09-01 |

## MLRO sign-off

Decision: ☐ Approve  ☐ Approve with conditions  ☐ Reject

Conditions / rationale: ______________________________

Name: ______________________  Date: ____________


## 4. Impact of a rulebook change, before sign-off

Proposal: let medium-risk applications auto-approve too. Which live decisions would change?

In [6]:
proposed = copy.deepcopy(rb)
proposed["risk_scoring"]["auto_approve_bands"] = ["low", "medium"]
for m in impact(cases, rb, proposed, AS_OF):
    print(f"{m['case_id']} {m['legal_name']}: {m['from']} → {m['to']}")
print("\nEverything else stays put, because a specific trigger (sanctions, missing documents, "
      "high-risk jurisdiction) still decides those cases.")

C002 Harbour Freight Traders Ltd: ANALYST_REVIEW → AUTO_APPROVE

Everything else stays put, because a specific trigger (sanctions, missing documents, high-risk jurisdiction) still decides those cases.


## 5. Overrides

In [7]:
from kyb.overrides import OverrideError
try:
    cli.override("output", "C005", "ANALYST_REVIEW", "OVR-03", "Client says it has moved its operations.", "A. Analyst", "ANALYST")
except OverrideError as e:
    print("Refused:", e)

ov = cli.override("output", "C002", "AUTO_APPROVE", "OVR-03",
                  "Ten-year trading history reviewed; sector rating overstates the risk.", "A. Analyst", "ANALYST")
print("Accepted:", ov["from"], "→", ov["to"], f"({ov['reason_code']}: {ov['reason']})")

Refused: Override on C005 refused: only the MLRO may override REJECT
Accepted: ANALYST_REVIEW → AUTO_APPROVE (OVR-03: Risk rating adjusted on a documented business rationale)


## 6. The audit log

In [8]:
print(audit.verify(Path("output/audit.jsonl"))[1])

# Quietly change one past decision and check again
p = Path("output/audit.jsonl")
original = p.read_text()
lines = original.splitlines()
e = json.loads(lines[4]); e["payload"]["outcome"] = "AUTO_APPROVE"; lines[4] = json.dumps(e)
p.write_text("\n".join(lines) + "\n")
print(audit.verify(p)[1])
_ = p.write_text(original)  # put it back

14 entries, chain intact
line 5: contents changed after it was written
